# Модуль 13.6 — Skills: выученный инструмент, своими руками на smolagents

Этот ноутбук — практика к лекции «Skills: выученный инструмент» (модуль 13.6 на сайте курса). Главный термин лекции — «skill — это инструмент, описанный словами и положенный в память агента»; здесь вы собираете механизм, который эти слова доставляет модели, — своими руками.

Сквозной герой — тот же, что в 11.5: агент-лесоруб в мок-лесу. Но чиним мы уже не tools: хорошие `move` / `gather` / `deposit` / `get_map` достаются в наследство готовыми. Симптом теперь другой — *действия* у агента есть, а *умения* нет: процедуру «пополни склад деревом» он каждый раз собирает заново и сдаёт полупустой рюкзак. Лекарство — skill: процедура, записанная markdown-файлом, которую агент подтягивает сам, когда задача совпала с описанием.

**Что вы получите на выходе:**

- мини-runtime скиллов поверх smolagents: skills лежат настоящими файлами `skills/<name>/SKILL.md`, в контекст агента попадает только витрина (`name` + `description`), тело подтягивается инструментом `load_skill`;
- progressive disclosure руками: Discovery → Activation → Execution, каждая стадия — отдельная ячейка с печатью;
- харнесс «без skill vs со skill» — разрыв цифрами, как в 11.5;
- линтер skill-lint по чек-листу A→H — правила как код.

**Карта ноутбука:**

- **Блок 1** — наследство 11.5 (мир и tools одной ячейкой) + симптом: агент без процедуры.
- **Блок 2 (ядро, keyless)** — skill-файлы, парсер, витрина, селектор, исполнение по скиллу, «сломай описание».
- **Блок 3 (ядро, keyless)** — линтер skill-lint A→H.
- **Блок 4 (опционально, нужен локальный сервер)** — живая модель + `load_skill`: progressive disclosure вживую.
- **Задачи** — stone-run, god-skill и отрыв триггера, доводка худого скилла, мостик в настоящую память агента.

Главное про запуск: ноутбук исполняется **целиком и без единого ключа** (`Run all`, без правок) за ~1–2 минуты. Интернет нужен один раз — поставить smolagents. Блок 4 делает мягкий пропуск (soft-skip), если локального сервера нет, — keyless-прогон остаётся зелёным. Артефакт модуля — прогнанный ноутбук, папка `skills/` с вашей библиотекой и (в Задаче 4) свой `SKILL.md` в памяти настоящего агента.

## Подготовка окружения

Ставим две библиотеки: `smolagents` (движок из Модуля 10 — из него берём `@tool`, `Tool` и агентов) и `openai` (клиент OpenAI-совместимых серверов — нужен `OpenAIServerModel` в Блоке 4; smolagents сам его не тянет).

Честная пометка: **для установки нужен интернет**. В Colab он есть всегда; в Kaggle включите `Notebook options → Internet → On` (нужен phone-verified аккаунт). После установки весь базовый трек работает без сети. API-ключи не нужны нигде: Блоку 4 достаточно локального сервера на вашей машине, без токенов.

In [ ]:
import sys, subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import smolagents  # noqa: F401
except Exception:
    _pip_install("smolagents")
    import smolagents  # noqa: F401

try:
    import openai  # noqa: F401  # нужен для OpenAIServerModel в Блоке 4
except Exception:
    _pip_install("openai")
    import openai  # noqa: F401

print("smolagents", smolagents.__version__, "| openai", openai.__version__)

## Блок 1. Наследство 11.5: мир и tools

В 11.5 вы строили это по шагам — имя по одной цели, `enum` в схеме, обучающая ошибка, конверт, кулдаун. Здесь весь финальный «хороший» набор приезжает одной ячейкой, дословно: мок-лес `Forest` (сетка 5×5, дерево на `(1, 2)` и `(3, 1)`, камень на `(2, 4)`, рюкзак на 5, склад на `(0, 0)`, кулдаун после каждого действия) и четыре инструмента `move` / `gather` / `deposit` / `get_map` с конвертом `{result, cooldown, state}` и обучающими ошибками `{code, message}`: `no_resource_here`, `inventory_full`, `not_at_storehouse`, `on_cooldown`.

Одно отличие от лекции: там `COOLDOWN = 1.0`, здесь — `0.3`, только чтобы `Run all` со всеми честными паузами укладывался в пару минут; правила те же. Если хотите освежить арку целиком — практика модуля 11.5 лежит рядом в homework-репо (ссылка в README).

In [ ]:
# Наследство 11.5: мок-лес и финальный «хороший» набор tools — дословно из модуля 11.5.
import json
import time

from smolagents import tool, Tool


class Forest:
    """Мок-лес: сетка 5x5, лесоруб, рюкзак, склад и кулдаун."""

    COOLDOWN = 0.3  # в лекции 1.0; здесь меньше, чтобы Run all занимал ~минуту, — правила те же

    def __init__(self):
        self.size = 5
        self.nodes = {(1, 2): "wood", (3, 1): "wood", (2, 4): "stone"}
        self.pos = (0, 0)        # где стоит лесоруб
        self.home = (0, 0)       # клетка склада
        self.backpack = {}       # например, {"wood": 3}
        self.cap = 5             # вместимость рюкзака
        self.stock = {}          # что уже сдано на склад
        self.busy_until = 0.0    # когда закончится кулдаун

    def cooldown_left(self):
        return max(0.0, self.busy_until - time.monotonic())

    def start_cooldown(self):
        self.busy_until = time.monotonic() + self.COOLDOWN

    def backpack_load(self):
        return sum(self.backpack.values())


forest = Forest()


def reset_forest():
    """Свежий мир перед каждым сценарием. Tools ниже смотрят на глобальную forest."""
    global forest
    forest = Forest()


def wait_ready():
    """Честно подождать конец кулдауна (наш харнесс, не часть контракта)."""
    time.sleep(forest.cooldown_left())


@tool
def get_map() -> dict:
    """Карта леса: позиция лесоруба, узлы ресурсов и клетка склада. Мир не меняет."""
    return {"pos": list(forest.pos), "home": list(forest.home),
            "nodes": [{"pos": list(p), "resource": r} for p, r in forest.nodes.items()]}


class GatherTool(Tool):
    name = "gather"
    description = (
        "Добыть один ресурс с клетки, на которой стоит лесоруб. "
        "Без resource берёт то, что есть на клетке; с resource — только ожидаемое."
    )
    inputs = {
        "resource": {
            "type": "string",
            "enum": ["wood", "stone"],
            "nullable": True,
            "description": "Ожидаемый ресурс: wood или stone. По умолчанию — любой.",
        }
    }
    output_type = "object"

    def forward(self, resource: str | None = None) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        node = forest.nodes.get(forest.pos)
        if node is None or (resource is not None and node != resource):
            return {"error": {"code": "no_resource_here",
                              "message": "На этой клетке нет нужного узла — "
                                         "найдите его через get_map и подойдите move."}}
        if forest.backpack_load() >= forest.cap:
            return {"error": {"code": "inventory_full",
                              "message": f"Рюкзак полон ({forest.cap}/{forest.cap}) — "
                                         "вернитесь на склад (0, 0) и позовите deposit."}}
        forest.backpack[node] = forest.backpack.get(node, 0) + 1
        forest.start_cooldown()
        return {"result": {"gathered": node, "amount": 1},
                "cooldown": Forest.COOLDOWN,
                "state": {"pos": list(forest.pos),
                          "backpack": dict(forest.backpack), "cap": forest.cap}}


class MoveTool(Tool):
    name = "move"
    description = (
        "Шаг на одну клетку в сторону direction. "
        "Зовите, когда до нужного узла или склада не хватает шага."
    )
    inputs = {
        "direction": {
            "type": "string",
            "enum": ["north", "south", "east", "west"],
            "description": "Куда шагнуть: north, south, east или west.",
        }
    }
    output_type = "object"

    def forward(self, direction: str) -> dict:
        wait = forest.cooldown_left()
        if wait > 0:
            return {"error": {"code": "on_cooldown",
                              "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
        dx, dy = {"north": (0, -1), "south": (0, 1),
                  "east": (1, 0), "west": (-1, 0)}[direction]
        x, y = forest.pos
        forest.pos = (min(max(x + dx, 0), forest.size - 1),
                      min(max(y + dy, 0), forest.size - 1))
        forest.start_cooldown()
        return {"result": {"pos": list(forest.pos)},
                "cooldown": Forest.COOLDOWN,
                "state": {"pos": list(forest.pos),
                          "backpack": dict(forest.backpack), "cap": forest.cap}}


@tool
def deposit() -> dict:
    """Сдать содержимое рюкзака на склад. Работает только на клетке склада (0, 0)."""
    wait = forest.cooldown_left()
    if wait > 0:
        return {"error": {"code": "on_cooldown",
                          "message": f"Лесоруб занят ещё {wait:.1f} с — подождите и повторите."}}
    if forest.pos != forest.home:
        return {"error": {"code": "not_at_storehouse",
                          "message": f"Склад на клетке (0, 0), а вы — на {forest.pos}. "
                                     "Дойдите до склада и повторите deposit."}}
    banked, forest.backpack = forest.backpack, {}
    for res, n in banked.items():
        forest.stock[res] = forest.stock.get(res, 0) + n
    forest.start_cooldown()
    return {"result": {"banked": banked},
            "cooldown": forest.COOLDOWN,
            "state": {"pos": list(forest.pos), "backpack": {}, "stock": forest.stock}}


move, gather = MoveTool(), GatherTool()
TOOLS = {"move": move, "gather": gather, "deposit": deposit, "get_map": get_map}

print("Наследство 11.5 собрано, tools:", list(TOOLS))
print("Карта:", json.dumps(get_map(), ensure_ascii=False))

### Симптом: действия есть, умения нет

Дадим агенту задачу «пополни склад деревом» — и ни слова о том, как это делается. Как и в 11.5, роль модели играет скрипт, чтобы цифры были воспроизводимы: он честно отыгрывает агента, который *не знает процедуры*. Рубит прямо на складе (ловит `no_resource_here` — ход потерян), добредает до дерева, делает два удара, решает «хватит», возвращается и сдаёт 2/5.

Заметьте: обучающие ошибки из 11.5 работают — агент не зациклился. Но они лечат *промах*, а не *незнание процедуры*: что рубить надо до `inventory_full`, а потом обязательно вернуться к складу, агент не знает — и в следующей сессии не вспомнит. Это ровно симптом из лекции.

In [ ]:
def _act(trace, tool_obj, note="", expected_error=None, **kwargs):
    """Один вызов tool с честным ожиданием кулдауна; шаг попадает в трейс."""
    wait_ready()
    res = tool_obj(**kwargs)
    err = res.get("error") if isinstance(res, dict) else None
    code = err["code"] if err else None
    args = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
    trace.append({"call": f"{tool_obj.name}({args})", "code": code,
                  "signal": code is not None and code == expected_error, "note": note})
    return res


def print_trace(trace):
    for i, t in enumerate(trace, 1):
        out = f"error {t['code']}" if t["code"] else "ok"
        if t["signal"]:
            out += " — сигнал перехода"
        note = f"  # {t['note']}" if t["note"] else ""
        print(f"  {i:>2}. {t['call']:<26} -> {out:<34}{note}")


def run_without_skill(verbose=True):
    """Симптом из лекции: старательный агент без процедуры."""
    reset_forest()
    trace = []
    _act(trace, gather, note="рубить прямо тут (а тут склад)")      # no_resource_here
    for d in ("east", "south", "south"):
        _act(trace, move, note="добрёл до дерева (1, 2)", direction=d)
    _act(trace, gather, note="удар топором", resource="wood")
    _act(trace, gather, note="ещё удар — и решил, что хватит", resource="wood")
    for d in ("west", "north", "north"):
        _act(trace, move, note="вернулся к складу", direction=d)
    load = forest.backpack_load()
    res = _act(trace, deposit, note="сдал что было")
    banked = sum(res["result"]["banked"].values())
    if verbose:
        print_trace(trace)
    errors = [t for t in trace if t["code"]]
    return {"сценарий": "без skill",
            "вызовы с error": f"{len(errors)} (впустую)",
            "рюкзак при deposit": f"{load}/{forest.cap}",
            "сдано": banked,
            "действий": len(trace)}


print("Задача: «Пополни склад деревом» — и ни слова о процедуре.")
print()
stats_no_skill = run_without_skill()
print()
for k, v in stats_no_skill.items():
    print(f"  {k}: {v}")

## Блок 2 (ядро, keyless). Skill: процедура словами в файле

Чиним. Записываем процедуру снабжения один раз — обычным markdown-файлом с двумя частями, как в лекции:

- **frontmatter** — паспорт: `name` (kebab-case) и `description` — «когда применять и что делает». Это самое важное поле файла: именно его, а не тело, читает агент, решая, подтянуть ли skill.
- **тело** — инструкция для модели: **шаги** (что делать по порядку), **грабли** (где наивная попытка ломается — те самые ошибки из 11.5, о которых агент теперь знает заранее), **проверка** (как понять, что готово).

Скиллов сразу два — `supply-run` (пополнить склад деревом) и `scout-map` (разведать лес): витрина из одного пункта была бы не выбором, а формальностью. Кладём их настоящими файлами `skills/<name>/SKILL.md` — skill живёт файлом, а значит, версионируется как код: git, PR, откат.

In [ ]:
import shutil
from pathlib import Path

SKILLS_DIR = Path("skills")
if SKILLS_DIR.exists():
    shutil.rmtree(SKILLS_DIR)   # чистый старт: повторный Run all не тащит хвосты прошлого прогона

SUPPLY_RUN = """\
---
name: supply-run
description: >-
  Применяй, когда на складе мало дерева и его нужно пополнить.
  Ведёт лесоруба к ближайшему дереву, рубит до полного рюкзака,
  возвращает к складу и сдаёт добычу.
---

# Supply run — пополнить склад деревом

Цель: положить на склад полный рюкзак дерева, не потеряв ходы
и не забыв вернуться.

## Шаги
1. `get_map` — найди ближайший узел `wood` и клетку склада `home`.
2. `move` к узлу по одной клетке (north / south / east / west).
3. `gather(resource="wood")`, пока не поймаешь `inventory_full`.
4. `move` обратно к складу `home`.
5. `deposit()`.
6. Складу нужно больше дерева — повтори с шага 1.

## Грабли
- Не зови `gather` на пустой клетке — сначала `move`, иначе `no_resource_here`.
- После каждого действия есть кулдаун — дождись `cooldown` секунд, не спамь.
- `deposit` работает только на клетке склада — иначе `not_at_storehouse`.

## Проверка
Готово, когда рюкзак пуст, а на складе прибавилось дерева.
"""

SCOUT_MAP = """\
---
name: scout-map
description: >-
  Применяй, когда нужно разведать лес: где узлы ресурсов, где склад,
  далеко ли до них. Собирает сводку по карте, ничего не меняя в мире.
---

# Scout map — разведать лес

Цель: сводка «что где лежит», не потратив ни одного хода-мутации.

## Шаги
1. `get_map` — позиция, склад, узлы.
2. Посчитай манхэттен-дистанцию от позиции до каждого узла и до склада.
3. Сведи отчёт: ресурс, клетка, дистанция; ближайший узел каждого ресурса отметь.

## Грабли
- Не двигайся и не добывай — разведка только читает; мутации жгут кулдаун.

## Проверка
В отчёте есть все узлы с карты и дистанции до них; мир не изменился.
"""


def write_skill(name, text, skills_dir=SKILLS_DIR):
    """Записать skills/<name>/SKILL.md (pathlib работает в Colab/Kaggle/локально)."""
    path = Path(skills_dir) / name / "SKILL.md"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path


def print_tree(skills_dir=SKILLS_DIR):
    """Дерево каталога skills/ — показать, что скиллы лежат файлами."""
    root = Path(skills_dir)
    print(f"{root.name}/")
    for sub in sorted(p for p in root.iterdir() if p.is_dir()):
        print(f"  {sub.name}/")
        for f in sorted(sub.iterdir()):
            print(f"    {f.name}")


write_skill("supply-run", SUPPLY_RUN)
write_skill("scout-map", SCOUT_MAP)
print_tree()

### Парсер и витрина: Discovery

Прочитаем файлы обратно. Парсер ниже — учебный, строк на двадцать пять: фронтматтер между первой парой `---`, `description:` умеет и значение в строке, и `>-` со свёрнутым блоком, всё после второй `---` — тело. Ошибка парсинга — `None`: такой файл просто не попадает в библиотеку (а линтер в Блоке 3 поймает это правилом A). В бою вы возьмёте готовый парсер фронтматтера; здесь важно один раз увидеть механику насквозь.

Из распарсенных скиллов собираем **витрину** — Discovery из лекции: по строке `- name: description` на каждый skill. Главное — в контексте агента постоянно живёт только это. Тел там нет: двадцать скиллов стоят двадцать коротких строк, а не двадцать простыней.

In [ ]:
def parse_skill(text):
    """SKILL.md -> {"name", "description", "body"}; ошибка парсинга -> None."""
    lines = text.splitlines()
    if not lines or lines[0].strip() != "---":
        return None
    try:
        end = lines.index("---", 1)          # закрывающая пара фронтматтера
    except ValueError:
        return None
    front, body = lines[1:end], "\n".join(lines[end + 1:]).strip()
    name, description, i = "", "", 0
    while i < len(front):
        line = front[i]
        if line.startswith("name:"):
            name = line.split(":", 1)[1].strip()
        elif line.startswith("description:"):
            value = line.split(":", 1)[1].strip()
            if value in (">-", ">"):         # свёрнутый блок: строки с отступом клеим пробелом
                folded = []
                while i + 1 < len(front) and front[i + 1].startswith(" "):
                    folded.append(front[i + 1].strip())
                    i += 1
                value = " ".join(folded)
            description = value
        i += 1
    return {"name": name, "description": description, "body": body}


def load_skills(skills_dir=SKILLS_DIR):
    """Обойти skills/*/SKILL.md; вернуть список распарсенных dict."""
    skills = []
    for path in sorted(Path(skills_dir).glob("*/SKILL.md")):
        skill = parse_skill(path.read_text(encoding="utf-8"))
        if skill:
            skills.append(skill)
    return skills


def skills_index(skills):
    """«Витрина»: только name + description. В контексте агента живёт только это."""
    return "\n".join(f"- {s['name']}: {s['description']}" for s in skills)


skills = load_skills()
assert len(skills) == 2, "на диске два скилла, оба должны распарситься"
print("Витрина (всё, что агент видит на старте):")
print()
print(skills_index(skills))

### Селектор: Activation

Пришла задача — кто решает, какой skill подтянуть? В бою — сама модель: читает витрину и сверяет её с задачей. Здесь, в keyless-режиме, её роль играет детерминированная функция — так же, как скриптовый «агент» играл модель в 11.5: совпадение значимых слов задачи со словами описания (сравниваем по префиксам, чтобы «деревом» и «дерева» считались одним словом), порог отсекает случайные совпадения.

Главное в этой функции — не формула, а то, чего она *не видит*: тело скилла ей недоступно, только `description`. Решение «подтянуть или нет» принимается по витрине — это и есть суть progressive disclosure.

In [ ]:
import re


def norm_words(text, prefix=4):
    words = re.findall(r"[а-яёa-z]+", text.lower())
    return {w[:prefix] for w in words if len(w) >= 4}


def match_score(task, description):
    return len(norm_words(task) & norm_words(description))


def select_skill(task, skills, threshold=2):
    scores = {s["name"]: match_score(task, s["description"]) for s in skills}
    best = max(scores, key=scores.get)
    chosen = best if scores[best] >= threshold else None
    return chosen, scores


TASK_WOOD = "Пополни склад деревом: нужен полный рюкзак дерева на складе."
TASK_SCOUT = "Разведай, где что лежит на карте леса."
TASK_STONE = "Пополни склад камнем."
TASK_JUNK = "Спой песню про лесоруба."

names = [s["name"] for s in skills]
print(f"{'задача':<52} " + " ".join(f"{n:>10}" for n in names) + "   выбран")
for task in (TASK_WOOD, TASK_SCOUT, TASK_STONE, TASK_JUNK):
    chosen, scores = select_skill(task, skills)
    row = " ".join(f"{scores[n]:>10}" for n in names)
    print(f"{task:<52} {row}   {chosen}")

chosen_w, scores_w = select_skill(TASK_WOOD, skills)
assert chosen_w == "supply-run" and scores_w["supply-run"] >= 3
assert scores_w["scout-map"] <= 1
assert select_skill(TASK_SCOUT, skills)[0] == "scout-map"
assert select_skill(TASK_STONE, skills)[0] == "supply-run"   # развести некому — задел Задачи 1
assert select_skill(TASK_JUNK, skills)[0] is None            # порог отсекает мусор

print()
print("«Пополни склад камнем» пока уезжает в supply-run: слова про склад совпали,")
print("а скилла про камень в библиотеке нет. Развести их — Задача 1.")
print("«Спой песню» не дотянула до порога — выбран None: селектор честно молчит.")

### Execution: исполняем шаги supply-run

Триггер сработал — тело подтянуто. Осталась третья стадия: исполнить шаги. В бою это делает модель; наш скриптовый исполнитель отыгрывает агента, который следует телу `supply-run` дословно: `get_map` → маршрут к ближайшему дереву → `gather(resource="wood")`, пока не поймает `inventory_full`, → маршрут к складу → `deposit`.

Обратите внимание на кадр из 11.5: единственная ошибка в трейсе — `inventory_full`, и это не промах, а **сигнал перехода** к шагу 4. Скилл прямо говорит: руби, «пока не поймаешь `inventory_full`» — ошибка встроена в процедуру как условие выхода из цикла.

In [ ]:
def _route(src, dst):
    """Маршрут по манхэттену: по одной клетке, сперва x, потом y."""
    (x, y), (tx, ty) = src, dst
    steps = ["east" if tx > x else "west"] * abs(tx - x)
    steps += ["south" if ty > y else "north"] * abs(ty - y)
    return steps


def run_with_skill(resource="wood", verbose=True):
    """Исполнить шаги supply-run; параметр resource — задел под stone-run (Задача 1)."""
    reset_forest()
    trace = []
    gmap = _act(trace, get_map, note="шаг 1: карта")
    pos, home = tuple(gmap["pos"]), tuple(gmap["home"])
    nodes = [tuple(n["pos"]) for n in gmap["nodes"] if n["resource"] == resource]
    target = min(nodes, key=lambda p: abs(p[0] - pos[0]) + abs(p[1] - pos[1]))
    for d in _route(pos, target):
        _act(trace, move, note=f"шаг 2: к узлу {target}", direction=d)
    while True:   # шаг 3: рубим, пока не inventory_full — ошибка это сигнал перехода
        res = _act(trace, gather, note="шаг 3: рубим",
                   expected_error="inventory_full", resource=resource)
        if "error" in res:
            break
    for d in _route(target, home):
        _act(trace, move, note="шаг 4: домой", direction=d)
    load = forest.backpack_load()
    res = _act(trace, deposit, note="шаг 5: сдаём")
    banked = sum(res["result"]["banked"].values())
    if verbose:
        print_trace(trace)
    errors = [t for t in trace if t["code"]]
    wasted = [t for t in errors if not t["signal"]]
    kind = "впустую" if wasted else "сигнал"
    return {"сценарий": f"со skill ({resource})",
            "вызовы с error": f"{len(errors)} ({kind})",
            "рюкзак при deposit": f"{load}/{forest.cap}",
            "сдано": banked,
            "действий": len(trace)}


print("Исполняем шаги supply-run:")
print()
stats_with_skill = run_with_skill()
print()
for k, v in stats_with_skill.items():
    print(f"  {k}: {v}")

### Разрыв цифрами: без skill vs со skill

Сведём оба прогона в одну таблицу. Как и в 11.5, смотрите не на абсолютные числа, а на **разрыв**: мир один, tools одни и те же, агент одинаково старается. Вся разница — в том, что второму дали процедуру словами.

In [ ]:
def harness_table(rows):
    """Харнесс как в 11.5: смотрите на разрыв."""
    cols = ["сценарий", "вызовы с error", "рюкзак при deposit", "сдано", "действий"]
    widths = [max(len(c), *(len(str(r[c])) for r in rows)) for c in cols]
    header = "  ".join(c.ljust(w) for c, w in zip(cols, widths))
    print(header)
    print("-" * len(header))
    for r in rows:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(cols, widths)))
    print("Ошибка со скиллом одна и ожидаемая: inventory_full — сигнал шага 3,")
    print("а не потеря. Разрыв — в рюкзаке при deposit и в сданном на склад.")


harness_table([stats_no_skill, stats_with_skill])

assert stats_no_skill["сдано"] == 2 and stats_no_skill["рюкзак при deposit"] == "2/5"
assert stats_no_skill["вызовы с error"] == "1 (впустую)"
assert stats_with_skill["сдано"] == 5 and stats_with_skill["рюкзак при deposit"] == "5/5"
assert stats_with_skill["вызовы с error"] == "1 (сигнал)"

### Описание — триггер, а не документация

Самый важный эксперимент ноутбука — зеркало «сломай описание» из домашки лекции. Портим у `supply-run` ровно одно поле — `description` становится «Помогает с игрой.» — а тело с его идеальными шагами не трогаем. Пересобираем витрину, даём ту же задачу — и скилл мёртв: до тела дело просто не доходит. Возвращаем точное описание — оживает.

Если после этого хочется проверять описания автоматически — правильно хочется: этим займётся линтер в Блоке 3.

In [ ]:
_TRIGGER = (
    "description: >-\n"
    "  Применяй, когда на складе мало дерева и его нужно пополнить.\n"
    "  Ведёт лесоруба к ближайшему дереву, рубит до полного рюкзака,\n"
    "  возвращает к складу и сдаёт добычу."
)
SUPPLY_RUN_VAGUE = SUPPLY_RUN.replace(_TRIGGER, "description: Помогает с игрой.")
assert SUPPLY_RUN_VAGUE != SUPPLY_RUN, "сломали именно описание, тело не тронуто"

write_skill("supply-run", SUPPLY_RUN_VAGUE)
chosen, scores = select_skill(TASK_WOOD, load_skills())
print("описание «Помогает с игрой.»:", scores, "-> выбран", chosen)
assert chosen is None and scores["supply-run"] == 0, "размытое описание = мёртвый skill"

write_skill("supply-run", SUPPLY_RUN)   # чиним описание
chosen, scores = select_skill(TASK_WOOD, load_skills())
print("описание вернули:            ", scores, "-> выбран", chosen)
assert chosen == "supply-run", "точное описание вернуло скилл к жизни"

print()
print("Тело не менялось ни на символ. Вся разница поведения — в одном поле description.")

## Блок 3 (ядро, keyless). Линтер skill-lint: A→H

В 11.5 чек-лист A→H судил контракт tool. Skill — тоже контракт, который читает модель, и к нему прикладывается такой же чек-лист. Восемь правил, каждое — функция с коротким вердиктом:

- **A** — frontmatter распарсился, `name` и `description` непустые;
- **B** — `name` в kebab-case, не длиннее 64 символов, без `claude`/`anthropic`;
- **C** — `description` не длиннее 1024 символов;
- **D** — `description` начинается с триггер-шаблона «Применяй, когда» / «Use when»;
- **E** — описание не размыто: без фраз чёрного списка («помогает с», «делает всё», «универсальный», «и вообще») и с запасом значимых слов;
- **F** — в теле есть раздел «Шаги» с двумя и более нумерованными пунктами;
- **G** — в теле есть раздел «Грабли» хотя бы с одним пунктом;
- **H** — в теле есть раздел «Проверка», и тело не длиннее 60 строк — оно для модели, без воды.

Лимиты в B и C — не выдумка: это ограничения формата skills у Claude Code. D и E охраняют триггер, F–H — тот самый триптих «шаги / грабли / проверка».

In [ ]:
KEBAB = re.compile(r"^[a-z0-9]+(-[a-z0-9]+)*$")
VAGUE_PHRASES = ("помогает с", "делает всё", "универсальный", "и вообще")
TRIGGERS = ("Применяй, когда", "Use when")
EMPTY = {"name": "", "description": "", "body": ""}


def _words(text):
    """Значимые слова: длина >= 4."""
    return [w for w in re.findall(r"[а-яёa-z]+", text.lower()) if len(w) >= 4]


def _section(body, title):
    """Строки раздела «## <title>» до следующего «## »; None — раздела нет."""
    found, out = False, []
    for line in body.splitlines():
        if line.startswith("## "):
            if found:
                break
            found = title.lower() in line.lower()
        elif found:
            out.append(line)
    return out if found else None


def rule_a(s, raw):
    ok = bool(s["name"]) and bool(s["description"])
    return ok, ("frontmatter распарсился, name и description непустые" if ok
                else "frontmatter не распарсился или name/description пустые")


def rule_b(s, raw):
    name = s["name"]
    ok = (bool(KEBAB.match(name)) and len(name) <= 64
          and "claude" not in name and "anthropic" not in name)
    return ok, ("name в kebab-case, <=64, без claude/anthropic" if ok
                else f"name {name!r} не проходит kebab-case/длину/стоп-слова")


def rule_c(s, raw):
    n = len(s["description"])
    return n <= 1024, f"description {n} символов (лимит 1024)"


def rule_d(s, raw):
    ok = s["description"].startswith(TRIGGERS)
    return ok, ("начинается с триггер-шаблона" if ok
                else "нет триггер-шаблона «Применяй, когда» / «Use when»")


def rule_e(s, raw):
    desc = s["description"].lower()
    bad = [p for p in VAGUE_PHRASES if p in desc]
    rich = len(_words(desc)) >= 5
    ok = not bad and rich
    if ok:
        note = "описание конкретное: без размытых фраз, слов хватает"
    elif bad:
        note = f"размытая фраза из чёрного списка: {bad[0]!r}"
    else:
        note = "меньше 5 значимых слов — триггеру не за что зацепиться"
    return ok, note


def rule_f(s, raw):
    sec = _section(s["body"], "шаги")
    steps = [ln for ln in sec or [] if re.match(r"\s*\d+[.)]", ln)]
    ok = sec is not None and len(steps) >= 2
    return ok, (f"раздел «Шаги»: {len(steps)} нумерованных пунктов" if ok
                else "нет раздела «Шаги» с >=2 нумерованными пунктами")


def rule_g(s, raw):
    sec = _section(s["body"], "грабли")
    items = [ln for ln in sec or [] if re.match(r"\s*([-*]|\d+[.)])\s", ln)]
    ok = sec is not None and len(items) >= 1
    return ok, (f"раздел «Грабли»: {len(items)} пунктов" if ok
                else "нет раздела «Грабли» хотя бы с одним пунктом")


def rule_h(s, raw):
    n = len(s["body"].splitlines())
    ok = _section(s["body"], "проверка") is not None and n <= 60
    return ok, (f"есть «Проверка», тело {n} строк (лимит 60)" if ok
                else "нет раздела «Проверка» или тело длиннее 60 строк")


RULES = [("A", rule_a), ("B", rule_b), ("C", rule_c), ("D", rule_d),
         ("E", rule_e), ("F", rule_f), ("G", rule_g), ("H", rule_h)]


def skill_lint(skill, raw=""):
    """-> (список (правило, ok, комментарий), счёт N из 8)."""
    s = skill or EMPTY
    rows = []
    for code, fn in RULES:
        ok, note = fn(s, raw)
        rows.append((code, ok, note))
    score = sum(1 for _, ok, _ in rows if ok)
    return rows, score


def report(title, rows, score):
    """Печать отчёта линтера по одному скиллу."""
    print(f"skill-lint: {title}")
    for code, ok, note in rows:
        print(f"  {code} {'ok  ' if ok else 'FAIL'} {note}")
    print(f"  счёт: {score}/8")


print("Линтер готов: 8 правил, вход — распарсенный skill плюс сырой текст файла.")

Прогоним на всех троих. Два героя обязаны собрать 8/8, а «Помогает с игрой.»-вариант интересен тем, *какие именно* буквы упадут: тело у него то же, что у героя, — провалиться могут только правила про описание.

In [ ]:
results = {}
for title, text in [("supply-run", SUPPLY_RUN),
                    ("scout-map", SCOUT_MAP),
                    ("supply-run (сломанное описание)", SUPPLY_RUN_VAGUE)]:
    rows, score = skill_lint(parse_skill(text), text)
    report(title, rows, score)
    print()
    results[title] = (rows, score)

assert results["supply-run"][1] == 8, "герой обязан собрать 8/8"
assert results["scout-map"][1] == 8, "второй герой тоже"
vague_failed = {code for code, ok, _ in results["supply-run (сломанное описание)"][0] if not ok}
assert vague_failed == {"D", "E"}, "у vague-варианта сломан ровно триггер, не тело"

print("Вердикт линтера совпал с экспериментом Блока 2: у vague-варианта провалены D и E —")
print("описание, — а телу (F, G, H) претензий нет. Мёртвым skill делает именно описание.")

## Блок 4 (опционально, нужен локальный сервер). Живая модель подтягивает skill сама

До сих пор Activation исполнял скрипт. Теперь — то, ради чего всё затевалось: отдать витрину **живой модели** и посмотреть, как она сама решает подтянуть тело.

Модель — целиком на вашей машине, без единого токена, тем же приёмом, что Блок 3 модуля 11.5: любой OpenAI-совместимый сервер из Модулей 6.1/6.2 — Ollama (`http://localhost:11434/v1`) или LM Studio (`http://localhost:1234/v1`) с instruct-моделью от ~7B, умеющей tool calling (например, `qwen2.5:7b`). Если сервер живёт по другому адресу, задайте переменные окружения `LOCAL_API_BASE` и `LOCAL_MODEL_ID` — ячейка проверит их первыми.

Блок **необязательный**: сервера нет — ячейки печатают причину и мягко пропускаются, keyless-прогон остаётся зелёным. В Colab/Kaggle локального сервера не бывает — там блок пропустится всегда; запускайте его на своей машине (вариант C из README).

In [ ]:
import os

import requests

CANDIDATES = [
    ("Ollama", "http://localhost:11434/v1"),
    ("LM Studio", "http://localhost:1234/v1"),
]
if os.environ.get("LOCAL_API_BASE"):
    CANDIDATES.insert(0, ("LOCAL_API_BASE", os.environ["LOCAL_API_BASE"]))

LOCAL_BASE = None    # адрес найденного сервера
LOCAL_MODEL = None   # выбранная модель

for server_name, base in CANDIDATES:
    try:
        r = requests.get(f"{base}/models", timeout=2)
        r.raise_for_status()
        ids = [item["id"] for item in r.json().get("data", [])]
    except Exception:
        continue
    chat_ids = [i for i in ids if "embed" not in i.lower()]   # embedding-модели не годятся
    if chat_ids:
        LOCAL_BASE = base
        LOCAL_MODEL = os.environ.get("LOCAL_MODEL_ID", chat_ids[0])
        print(f"{server_name}: сервер найден на {base} -> модель {LOCAL_MODEL}")
        break

if LOCAL_BASE is None:
    print("Локальный сервер не найден -> Блок 4 пропущен (keyless-прогон это норма).")

### Пятый tool: `load_skill`

Как доставить модели витрину и дать способ подтянуть тело? Витрину кладём в instructions агента (system-слой) вместе с правилом: «задача совпала с описанием умения — сначала `load_skill(имя)`, затем следуй шагам». А подтягивание — отдельный инструмент `load_skill(name)`: по имени возвращает тело скилла строкой, на неизвестное имя отвечает обучающей ошибкой со списком доступных имён — «enum в форме ошибки» из 11.5, теперь для библиотеки умений.

Дальше — `ToolCallingAgent` с пятью инструментами и задача «Пополни склад деревом». После прогона проверяем две вещи **по фактам, а не по словам агента**: были ли вызовы `load_skill` — по логам `agent.memory`; что реально легло на склад — по `forest.stock`.

In [ ]:
class LoadSkillTool(Tool):
    """Execution по требованию: подтянуть тело скилла, когда триггер сработал."""

    name = "load_skill"
    description = (
        "Загрузить полный текст умения (skill) из библиотеки по имени. "
        "Зови, когда описание умения в списке совпало с задачей."
    )
    inputs = {"name": {"type": "string",
                       "description": "Имя умения из списка, например supply-run."}}
    output_type = "any"   # тело строкой либо конверт с обучающей ошибкой

    def __init__(self, skills_dir=SKILLS_DIR):
        super().__init__()
        self.skills_dir = Path(skills_dir)

    def forward(self, name: str):
        path = self.skills_dir / name / "SKILL.md"
        if not path.is_file():
            known = ", ".join(sorted(p.parent.name
                                     for p in self.skills_dir.glob("*/SKILL.md")))
            return {"error": {"code": "unknown_skill",
                              "message": f"Нет такого умения. Доступны: {known}."}}
        return parse_skill(path.read_text(encoding="utf-8"))["body"]


def build_instructions(skills):
    """System-слой агента: роль, правила мира кратко, витрина, правило подтягивания."""
    return (
        "Ты — лесоруб в мок-лесу 5x5 (x и y от 0 до 4). Склад — клетка (0, 0), "
        "рюкзак вмещает 5 ресурсов. Действуй только инструментами. Ошибка от "
        "инструмента — не провал, а подсказка: читай её code и message; "
        "on_cooldown значит «занят долю секунды» — повтори тот же вызов.\n\n"
        "Библиотека умений (name: description):\n"
        + skills_index(skills) + "\n\n"
        "Если задача совпала с описанием умения — сначала вызови load_skill(имя), "
        "затем следуй шагам из его текста."
    )


def load_skill_calls(agent):
    """Вызовы load_skill из памяти агента: логам верим больше, чем словам."""
    calls = []
    for step in agent.memory.steps:
        for tc in getattr(step, "tool_calls", None) or []:
            if tc.name == "load_skill":
                calls.append(tc.arguments)
    return calls


LIVE_TASK = "Пополни склад деревом: нужен полный рюкзак дерева на складе."


def run_live(task):
    """Живой прогон с load_skill; без сервера или при сбое -> мягкий пропуск."""
    if LOCAL_BASE is None:
        print("Блок 4 пропущен: локальный сервер не найден "
              "(Ollama :11434 / LM Studio :1234). Для keyless-прогона это норма.")
        return "skipped"
    try:
        from smolagents import OpenAIServerModel, ToolCallingAgent

        reset_forest()
        skills_now = load_skills()
        agent = ToolCallingAgent(
            tools=[move, gather, deposit, get_map, LoadSkillTool()],
            model=OpenAIServerModel(model_id=LOCAL_MODEL, api_base=LOCAL_BASE,
                                    api_key="local"),  # заглушка: ключ не проверяется
            max_steps=25,
            instructions=build_instructions(skills_now),
        )
        agent.run(task)
        print()
        print("Вызовы load_skill по логам agent.memory:", load_skill_calls(agent) or "не было")
        print("Склад после прогона (forest.stock):", forest.stock)
        return "ran"
    except Exception as e:
        print("Живой прогон не прошёл -> мягкий пропуск:", repr(e))
        return "skipped"


live_status = run_live(LIVE_TASK)

### «Сломай описание» — теперь с живой моделью

Повторим эксперимент Блока 2, но судьёй будет модель. Ломаем `description` на диске, пересобираем витрину и агента, даём ту же задачу: `load_skill`, скорее всего, вызван не будет — по размытому «Помогает с игрой.» триггер не срабатывает. Файл возвращаем на место в любом случае (`finally`).

Оговорка честности: сильная модель может решить задачу и без скилла — перепридумает процедуру на ходу. Разница видна в потраченных шагах и в том, тянулось ли тело; слабая модель без скилла чаще останавливается на полупустом рюкзаке — ровно как наш скрипт в Блоке 1.

In [ ]:
if LOCAL_BASE is None:
    print("Пропуск: локального сервера нет — эксперимент со сломанным описанием не запускаем.")
else:
    write_skill("supply-run", SUPPLY_RUN_VAGUE)
    try:
        print("Описание supply-run сломано («Помогает с игрой.») — та же задача:")
        print()
        run_live(LIVE_TASK)
    finally:
        write_skill("supply-run", SUPPLY_RUN)
        print()
        print("Описание вернули: файл supply-run снова точный.")

## Задачи

Теперь стройте библиотеку сами. Задачи 1–3 работают **на моке и селекторе** из Блоков 1–3 — ключи не нужны; Задача 4 уносит результат в вашего настоящего агента. Каждая задача уже содержит рабочий образец (ноутбук остаётся зелёным на `Run all`); ваша работа — разобрать образец, изменить под себя и прогнать снова.

Подход тот же, что в 11.5: запустите готовые ячейки, убедитесь, что цифры сходятся, а затем поэкспериментируйте — поменяйте формулировки описаний, тексты задач, пороги и посмотрите, как реагируют селектор и линтер.

### Задача 1. Второй рабочий skill: `stone-run`

В Блоке 2 «Пополни склад камнем» уезжала в `supply-run` — потому что разводить было некому. Напишите `stone-run` по образцу `supply-run`: камень в этом лесу один, узел `(2, 4)`.

Образец ниже — полный рабочий вариант, и все три проверки автоматические: селектор разводит «деревянную» и «каменную» задачи по разным скиллам; линтер даёт 8/8; скриптовый исполнитель сдаёт камень — `run_with_skill` уже принимает ресурс параметром, ему всё равно, дерево или камень.

Дальше — сами: поменяйте формулировку `description` и найдите вариант, на котором селектор перестаёт разводить задачи (подсказка: убрать слово «камня» — самый короткий способ убить триггер).

In [ ]:
STONE_RUN = """\
---
name: stone-run
description: >-
  Применяй, когда на складе мало камня и его нужно пополнить.
  Ведёт лесоруба к каменному узлу, добывает до полного рюкзака,
  возвращает к складу и сдаёт добычу.
---

# Stone run — пополнить склад камнем

Цель: положить на склад полный рюкзак камня, не потеряв ходы
и не забыв вернуться.

## Шаги
1. `get_map` — найди узел `stone` (в этом лесу он один — `(2, 4)`) и склад `home`.
2. `move` к узлу по одной клетке (north / south / east / west).
3. `gather(resource="stone")`, пока не поймаешь `inventory_full`.
4. `move` обратно к складу `home`.
5. `deposit()`.

## Грабли
- Не зови `gather` на пустой клетке — сначала `move`, иначе `no_resource_here`.
- После каждого действия есть кулдаун — дождись `cooldown` секунд, не спамь.
- `deposit` работает только на клетке склада — иначе `not_at_storehouse`.

## Проверка
Готово, когда рюкзак пуст, а на складе прибавилось камня.
"""

write_skill("stone-run", STONE_RUN)
skills3 = load_skills()
print("Витрина выросла:")
print(skills_index(skills3))
print()

for task in (TASK_WOOD, TASK_STONE):
    chosen, scores = select_skill(task, skills3)
    print(f"{task:<42} {scores} -> {chosen}")
assert select_skill(TASK_WOOD, skills3)[0] == "supply-run"
assert select_skill(TASK_STONE, skills3)[0] == "stone-run"
print("Селектор разводит: дерево -> supply-run, камень -> stone-run.")
print()

rows, score = skill_lint(parse_skill(STONE_RUN), STONE_RUN)
report("stone-run", rows, score)
assert score == 8, "stone-run обязан собрать весь чек-лист"
print()

print("Исполняем шаги stone-run (тот же исполнитель, ресурс параметром):")
print()
stats_stone = run_with_skill(resource="stone")
print()
for k, v in stats_stone.items():
    print(f"  {k}: {v}")
assert stats_stone["сдано"] == 5, "полный рюкзак камня на складе"

### Задача 2. God-skill: нечёткий триггер и отрыв

Зеркало «god-tool → узкие tools» из 11.5, теперь на уровне процедур. Вот готовый антигерой `forest-master` — «применяй для любой работы в лесу»: добыть, разведать, сдать и навести порядок. Тело-простыня, раздела «Проверка» нет.

Мера беды — **отрыв триггера** (margin): скор лучшего скилла минус скор второго. Узкий скилл выигрывает свою задачу с запасом; god-skill спорит со всеми понемногу — отрывы схлопываются, а разведку он и вовсе перехватывает — со счётом 2:2. При равенстве наш селектор берёт первого по алфавиту, и это само по себе звоночек: отрыв 0 — значит выбор скилла превратился в лотерею.

Чтобы цифры не зависели от того, записали ли вы `stone-run` в Задаче 1, тройку скиллов соберём в памяти — из текстов, в том же алфавитном порядке, в каком их прочитал бы `load_skills`. Расщепление здесь — просто не класть god-skill в библиотеку; на диске это было бы `rm -r skills/forest-master`.

In [ ]:
FOREST_MASTER = """\
---
name: forest-master
description: >-
  Применяй для любой работы в лесу: добыть дерево и камень, разведать
  карту, сдать всё на склад и вообще навести порядок.
---

# Forest master — вся работа в лесу

Один скилл на всё: добыча, разведка, сдача, порядок.

## Шаги
1. `get_map` — посмотри, что где.
2. Нужно дерево — иди к дереву и зови `gather`, пока не надоест.
3. Нужен камень — то же самое у камня.
4. Нужна разведка — посмотри карту ещё раз и перескажи.
5. Что-то добыл — когда-нибудь сдай на склад через `deposit`.
6. Если что-то пошло не так — попробуй ещё раз что-нибудь другое.
7. Наведи порядок: сходи туда, где давно не был.
"""


def margin(scores):
    """Отрыв триггера: лучший скор минус второй."""
    top = sorted(scores.values(), reverse=True)
    return top[0] - (top[1] if len(top) > 1 else 0)


god_lib = sorted((parse_skill(t) for t in (SUPPLY_RUN, SCOUT_MAP, FOREST_MASTER)),
                 key=lambda s: s["name"])
narrow_lib = [s for s in god_lib if s["name"] != "forest-master"]

for label, lib in (("с god-skill", god_lib), ("без god-skill", narrow_lib)):
    for task in (TASK_WOOD, TASK_SCOUT):
        chosen, scores = select_skill(task, lib)
        print(f"{label:<14} {task[:42]:<44} {scores} -> {chosen}, отрыв {margin(scores)}")
    print()

c_god, s_god = select_skill(TASK_SCOUT, god_lib)
assert c_god == "forest-master" and margin(s_god) == 0, "god-skill перехватил разведку: отрыв 0"
c_nar, s_nar = select_skill(TASK_SCOUT, narrow_lib)
assert c_nar == "scout-map" and margin(s_nar) > 0, "убрали god-skill — разведка вернулась к scout-map"
_, sw_god = select_skill(TASK_WOOD, god_lib)
_, sw_nar = select_skill(TASK_WOOD, narrow_lib)
assert margin(sw_nar) > margin(sw_god), "отрыв supply-run без god-skill тоже вырос"
print("Расщепление лечит: оба узких скилла выигрывают свои задачи с большим отрывом.")
print()

rows, score = skill_lint(parse_skill(FOREST_MASTER), FOREST_MASTER)
report("forest-master", rows, score)
assert score <= 4, "линтер согласен: у god-skill сломан триггер и нет проверки"

### Задача 3. Довести «худой» skill до 8/8

Симметрия Задачи 4 из 11.5: там вы доводили худую спеку tool, здесь — худой skill. Вот `wood-run-v1`: описание «Помогает с деревом.», шаги одной строкой, ни «Граблей», ни «Проверки». Линтер честно перечисляет, что чинить.

Образец доводки — `wood-run-v2`: триггер-шаблон в описании, шаги пронумерованы, грабли и проверка на месте. Зафиксируйте «до/после» и поэкспериментируйте: верните какую-нибудь правку назад — например, сотрите раздел «Грабли» — и посмотрите, какая буква упадёт.

In [ ]:
WOOD_RUN_V1 = """\
---
name: wood-run-v1
description: Помогает с деревом.
---

# Wood run v1

Сходи к дереву, наруби полный рюкзак и сдай на склад: get_map, move, gather, deposit.
"""

rows_v1, score_v1 = skill_lint(parse_skill(WOOD_RUN_V1), WOOD_RUN_V1)
report("wood-run-v1 (до)", rows_v1, score_v1)
print()

WOOD_RUN_V2 = """\
---
name: wood-run-v2
description: >-
  Применяй, когда нужно сходить за деревом: добыть полный рюкзак
  и сдать его на склад одним рейсом.
---

# Wood run v2 — рейс за деревом

Цель: один рейс — полный рюкзак дерева на складе.

## Шаги
1. `get_map` — найди ближайший узел `wood` и клетку склада `home`.
2. `move` к узлу по одной клетке.
3. `gather(resource="wood")` до ошибки `inventory_full`.
4. `move` к складу `home` и позови `deposit()`.

## Грабли
- `gather` на пустой клетке даёт `no_resource_here` — сначала дойди до узла.
- Между действиями кулдаун — дождись `cooldown` секунд, не спамь.

## Проверка
Рюкзак пуст, на складе стало больше дерева.
"""

rows_v2, score_v2 = skill_lint(parse_skill(WOOD_RUN_V2), WOOD_RUN_V2)
report("wood-run-v2 (после)", rows_v2, score_v2)
print()

print(f"Рост: {score_v1}/8 -> {score_v2}/8")
assert score_v1 <= 3, "худой скилл и должен проседать"
assert score_v2 == 8, "доведённый скилл собирает весь чек-лист"
print()
print("Кстати, wood-run-v2 по смыслу дублирует supply-run — в настоящей библиотеке")
print("вы бы оставили один из них; здесь пара нужна, чтобы зафиксировать «до/после».")

### Задача 4 (мостик в бой). Тот же skill — в настоящую память агента

Мини-runtime из этого ноутбука — учебная модель механизма, который в боевых агентах уже встроен. Финальный шаг домашки (по секции «Домашнее задание» лекции) — положить свой skill в настоящую память агента и увидеть progressive disclosure без всяких скриптов:

1. Возьмите свой `supply-run` или `stone-run` — либо боевого двойника из своего проекта: «оформить commit по нашему стилю», «прогнать линтер и починить типовые ошибки».
2. Положите файл в память агента. У Claude Code это `~/.claude/skills/<name>/SKILL.md` (личный) или `.claude/skills/<name>/SKILL.md` (в проекте); у другого агента — его каталог skills.
3. Дайте задачу, совпадающую с описанием, — агент объявит «Using \<skill\>…» и пойдёт по шагам.
4. Сломайте описание («помогает с игрой»), повторите ту же задачу — skill не подтянется. Верните точное — подтянется снова. Тело вы не трогали: тот же эксперимент, что вы дважды прогнали здесь, только судья — боевой агент.

Критерии приёма — из лекции, проверяете сами: skill подтягивается по точному описанию под релевантную задачу; при размытом — не подтягивается; в теле есть шаги, грабли и проверка.

Ячейка ниже соберёт заготовку: скопирует ваши скиллы с диска в `export-for-claude/` — останется перенести папку скилла в `~/.claude/skills/`. Проверка этой задачи происходит у вас в агенте, не в ноутбуке.

In [ ]:
EXPORT_DIR = Path("export-for-claude")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

exported = []
for path in sorted(SKILLS_DIR.glob("*/SKILL.md")):
    target = EXPORT_DIR / path.parent.name / "SKILL.md"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(path.read_text(encoding="utf-8"), encoding="utf-8")
    exported.append(path.parent.name)

print("Заготовка собрана: export-for-claude/")
for name in exported:
    print(f"  {name}/SKILL.md")
print()
print("Дальше на своей машине, например для supply-run:")
print("  mkdir -p ~/.claude/skills && cp -r export-for-claude/supply-run ~/.claude/skills/")
print("Проверка — у себя в агенте: задача под описание -> «Using supply-run…».")

## Что дальше

Вы собрали руками весь путь скилла: файл на диске → витрина в контексте → триггер по описанию → подтянутое тело → исполненные шаги — и дважды убедились, что описание важнее тела. Дальше по треку «tools → skills → MCP»:

- **Модуль 13. Hermes** — самообучающийся агент замыкает петлю: такие же markdown-файлы (шаги, грабли, проверка) он пишет и переписывает себе сам после решённых задач. Ваш `supply-run` — ровно то, что Hermes сгенерировал бы себе после пятого похода за деревом.
- **Модуль 14.5. MCP** — третий примитив: как один и тот же инструмент отдать по стандартному протоколу любому агенту, а не только своему.

**Критерии приёма (проверяете сами):**

- ноутбук прогнан целиком (`Run all`) keyless без ошибок;
- `stone-run` написан: селектор разводит дерево и камень, линтер даёт 8/8, исполнитель сдаёт 5/5 камня (Задача 1);
- на god-skill показан малый отрыв триггера, после расщепления отрыв вырос (Задача 2);
- «худой» `wood-run-v1` доведён до 8/8, «до/после» зафиксированы (Задача 3);
- свой skill лежит в памяти вашего агента, подтягивается по точному описанию и молчит при размытом (Задача 4).

Артефакт для сдачи — публичная ссылка на прогнанный ноутбук, в чат курса как `[Модуль 13.6, ДЗ] {ссылка}`.

Если так — домашка сдана, преподаватель не нужен.